# CausalBench K562 + RPE1 — Dataset Audit

**Goal:** (1) verify the *actual* files, expression-value format and perturbation labels; (2) produce a `cell_line × perturbation × number_of_cells` table for selecting TDA pilot experiments.

**Instructions:** Run cells in order in Jupyter or Google Colab. **Do not assume** that the counts from the CausalBench publication exactly match a third-party Hugging Face mirror. This notebook downloads only the file(s) you explicitly select. For large H5AD files, it reads `.obs` metadata in backed mode rather than loading the full expression matrix.

Dataset sources:
- `huggingbahl21/causalbench-k562`
- `huggingbahl21/causalbench-rpe1`

If the repository cannot be listed, check whether it is private, removed, misspelled or requires a Hugging Face login. The repository names and schemas have not been independently verified from this environment.

## 1. Install dependencies

In [ ]:
%pip install -q huggingface_hub anndata pyarrow pandas numpy scipy

## 2. List *actual* files in each Hugging Face dataset

This step does **not** download the large expression matrices. Look for `.h5ad`, `.parquet`, `.csv`, `.tsv`, `.npz`, `.npy` and README files. Note the folder names and sizes.

In [ ]:
from huggingface_hub import HfApi
import pandas as pd

REPOS = {
    "K562": "huggingbahl21/causalbench-k562",
    "RPE1": "huggingbahl21/causalbench-rpe1",
}
api = HfApi()
inventory_rows = []

for cell_line, repo in REPOS.items():
    print(f"\n### {cell_line}: {repo}")
    try:
        for item in api.list_repo_tree(repo_id=repo, repo_type="dataset", recursive=True):
            path = item.path
            if hasattr(item, "size"):  # files, not folder entries
                size_mb = (item.size or 0) / 1024**2
                inventory_rows.append({"cell_line": cell_line, "path": path, "size_MB": round(size_mb, 2)})
                print(f"  {path} ({size_mb:.2f} MB)")
    except Exception as exc:
        print(f"  ERROR: {type(exc).__name__}: {exc}")
        print("  Check the exact link, repo visibility, or Hugging Face authentication.")

inventory = pd.DataFrame(inventory_rows, columns=["cell_line", "path", "size_MB"])
display(inventory)
inventory.to_csv("causalbench_repo_inventory.csv", index=False)
print("Saved: causalbench_repo_inventory.csv")

## 3. Choose files to audit, then download ONLY those files

**Edit `SELECTED_FILES` based on the inventory above.** Each entry must be a path exactly as printed by Step 2 (folder prefixes included). Use multiple paths if the dataset is split into several `.parquet` or `.csv` files. H5AD typically contains both expression data (`X`, `layers`) and cell metadata (`obs`). A separate `.npy` expression array alone is insufficient for counting perturbations—you need the corresponding cell-level labels file.

**Important:** Do not blindly download all files when a repository is many gigabytes.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

# EDIT these paths after seeing Step 2's inventory.
# Example only: ["path/to/filename.h5ad"] (not an actual confirmed file name).
SELECTED_FILES = {
    "K562": [],
    "RPE1": [],
}

LOCAL_FILES = {"K562": [], "RPE1": []}
for cell_line, relative_paths in SELECTED_FILES.items():
    repo = REPOS[cell_line]
    for relative_path in relative_paths:
        try:
            local_path = hf_hub_download(
                repo_id=repo, filename=relative_path, repo_type="dataset"
            )
            LOCAL_FILES[cell_line].append(Path(local_path))
            print(f"{cell_line}: {relative_path} -> {local_path}")
        except Exception as exc:
            print(f"{cell_line}: could not download {relative_path}: {exc}")

if not any(LOCAL_FILES.values()):
    print("No data files selected. Edit SELECTED_FILES above and rerun this cell.")

## 4. Inspect format, columns and sample expression values

H5AD: prints the data shape, `.obs` metadata fields, `.var` gene names and sampled values from `X`.  Parquet/CSV: prints column names and a few **sample rows only**; these may contain metadata and/or expression, depending on the mirror. NPY/NPZ: prints array shapes and keys; a separate label file may be necessary.

*Negative values or noninteger values are clues—not proof—of a particular normalization method. Confirm the source documentation before transforming the expression matrix.*

In [ ]:
import numpy as np
import pandas as pd

def kind(path):
    name = path.name.lower()
    if name.endswith(".h5ad"): return "h5ad"
    if name.endswith(".parquet"): return "parquet"
    if name.endswith((".csv", ".csv.gz")): return "csv"
    if name.endswith((".tsv", ".tsv.gz")): return "tsv"
    if name.endswith(".npz"): return "npz"
    if name.endswith(".npy"): return "npy"
    return "other"

def audit_file(path):
    mode = kind(path)
    print(f"\nFILE: {path.name}  [{mode}]")
    if mode == "h5ad":
        import anndata as ad
        from scipy import sparse
        a = ad.read_h5ad(path, backed="r")
        try:
            print("shape [cells, genes]:", a.shape)
            print("obs columns:", list(a.obs.columns))
            print("first gene names:", list(a.var_names[:8]))
            print("layers:", list(a.layers.keys()))
            print("obs example (first 3 cells):")
            display(a.obs.head(3))
            sample = a.X[:min(100,a.n_obs), :min(100,a.n_vars)]
            if sparse.issparse(sample): sample = sample.toarray()
            sample = np.asarray(sample)
            if sample.size:
                finite = sample[np.isfinite(sample)]
                print("expression X sample: shape", sample.shape,
                      "dtype", sample.dtype, "min/max", (finite.min(), finite.max()) if finite.size else "n/a",
                      "zero fraction", round(float(np.mean(sample == 0)), 3),
                      "near-integer fraction", round(float(np.mean(np.isclose(sample, np.round(sample)))), 3))
        finally:
            a.file.close()
    elif mode == "parquet":
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        print("rows:", pf.metadata.num_rows)
        print("columns:", pf.schema_arrow.names[:50])
        cols = pf.schema_arrow.names[:min(8, len(pf.schema_arrow.names))]
        first = next(pf.iter_batches(batch_size=3, columns=cols), None)
        if first is not None: display(first.to_pandas())
    elif mode in ("csv", "tsv"):
        sep = "," if mode == "csv" else "\t"
        sample = pd.read_csv(path, sep=sep, nrows=3)
        print("columns (first 50):", list(sample.columns[:50]))
        display(sample)
    elif mode == "npy":
        arr = np.load(path, mmap_mode="r", allow_pickle=False)
        print("shape:", arr.shape, "dtype:", arr.dtype)
    elif mode == "npz":
        with np.load(path, allow_pickle=False) as z:
            print("NPZ array keys:", z.files)
            print("Individual arrays not loaded to avoid accidental memory exhaustion.")
    else:
        print("Non-tabular or unsupported type. Inspect metadata/README manually.")

for cell_line, paths in LOCAL_FILES.items():
    print(f"\n{'='*18} {cell_line} {'='*18}")
    for path in paths:
        audit_file(path)

## 5. Count cells for each perturbation (the key deliverable)

**Before running:** inspect Step 4 and set the **exact** perturbation-label column for each cell line. Often it will look like `intervention`, `perturbation` or `target_gene`, but names are not guaranteed. The selected column must contain **one label per cell**, not measured gene names, or a column of already aggregated statistics. For `.h5ad`, choose a column of `.obs`; for `.parquet`/`.csv`, choose a column from the table.

**Do not assign controls automatically.** The printed counts will show the exact label representation (e.g., `non-targeting`, `control`, or other). Adjust `CONTROL_LABELS` after inspecting those values. Missing labels are reported separately, not automatically treated as controls.

In [ ]:
from collections import Counter

# EDIT with the verified exact metadata column from Step 4.
LABEL_COLUMN = {"K562": None, "RPE1": None}

# EDIT with the exact control label(s) observed in the printed results.
CONTROL_LABELS = {"K562": set(), "RPE1": set()}


def labels_from_file(path, col):
    mode = kind(path)
    if mode == "h5ad":
        import anndata as ad
        a = ad.read_h5ad(path, backed="r")
        try:
            if col not in a.obs.columns:
                raise KeyError(f"{col!r} is not in .obs; options: {list(a.obs.columns)}")
            labels = a.obs[col].copy()
        finally:
            a.file.close()
        yield labels
    elif mode == "parquet":
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        if col not in pf.schema_arrow.names:
            raise KeyError(f"{col!r} not in Parquet columns: {pf.schema_arrow.names}")
        for batch in pf.iter_batches(batch_size=50000, columns=[col]):
            yield batch.to_pandas()[col]
    elif mode in ("csv", "tsv"):
        sep = "," if mode == "csv" else "\t"
        for chunk in pd.read_csv(path, sep=sep, usecols=[col], chunksize=50000):
            yield chunk[col]
    else:
        raise ValueError(f"{path.name}: use a separate cell-level label CSV/Parquet/H5AD file")


def clean_label_series(series):
    return series.astype("string").str.strip().fillna("<MISSING>")

count_rows = []
for cell_line, paths in LOCAL_FILES.items():
    label_col = LABEL_COLUMN[cell_line]
    if label_col is None:
        print(f"{cell_line}: LABEL_COLUMN not set; skipped.")
        continue
    if not paths:
        print(f"{cell_line}: no files downloaded; skipped.")
        continue
    counter = Counter()
    for path in paths:
        for labels in labels_from_file(path, label_col):
            counter.update(clean_label_series(labels).value_counts(dropna=False).to_dict())
    for perturbation, n_cells in counter.items():
        count_rows.append({"cell_line": cell_line,
                           "perturbation": str(perturbation),
                           "n_cells": int(n_cells),
                           "is_control": str(perturbation) in CONTROL_LABELS[cell_line]})
    print(f"\n{cell_line}: {sum(counter.values()):,} rows, {len(counter):,} distinct labels")
    print("Top 20 labels:", counter.most_common(20))

counts = pd.DataFrame(count_rows, columns=["cell_line","perturbation","n_cells","is_control"])
if not counts.empty:
    counts = counts.sort_values(["cell_line","n_cells"], ascending=[True,False])
    counts.to_csv("causalbench_perturbation_cell_counts.csv", index=False)
    display(counts.head(30))
    print("Saved: causalbench_perturbation_cell_counts.csv")
else:
    print("No counts yet. Fill in SELECTED_FILES and LABEL_COLUMN first.")

## 6. Select a **candidate** TDA pilot set

Start from perturbations with enough cells for **matched subsampling**; the threshold is exploratory. Include multiple guides/batches where metadata permit, and do not equate high cell count with reliable biological perturbation. **Only run after `CONTROL_LABELS` has been verified and set.**

In [ ]:
MIN_CELLS = 100   # exploratory threshold, revise after inspecting the count distribution

if not counts.empty:
    missing_controls = [line for line in counts.cell_line.unique() if not CONTROL_LABELS[line]]
    if missing_controls:
        print("Stop: verify and enter the actual control labels for:", missing_controls)
        print("Rerun Step 5 to refresh the is_control column, then run Step 6.")
    else:
        pilot = counts.loc[
            (counts.n_cells >= MIN_CELLS) &
            (~counts.is_control) &
            (counts.perturbation != "<MISSING>")
        ].copy()
        pilot.to_csv("causalbench_candidate_perturbations.csv", index=False)
        display(pilot.groupby("cell_line", group_keys=False).head(20))
        print("Saved: causalbench_candidate_perturbations.csv")

        # Shared label strings only: different gene-ID conventions require harmonization.
        wide = pilot.pivot_table(index="perturbation", columns="cell_line", values="n_cells", aggfunc="sum")
        shared = wide.dropna(subset=["K562", "RPE1"]) if {"K562","RPE1"}.issubset(wide.columns) else pd.DataFrame()
        if not shared.empty:
            shared.sort_values(["K562","RPE1"], ascending=False).to_csv("causalbench_shared_perturbations.csv")
            display(shared.head(20))
            print("Saved: causalbench_shared_perturbations.csv")
else:
    print("Run Step 5 successfully before creating the shortlist.")

## Interpretation checklist

1. **Rows are cells?** Check that each label belongs to one cell. A DE-results table with one row per `(perturbation, gene)` is *not* suitable for this count.
2. **Controls really identified?** Inspect the unique label values and exact metadata meaning; do not assume missing labels are controls.
3. **Guide/batch fields?** Record them if present. Multiple guides may target the same gene, and batch effects can mimic perturbation differences.
4. **Expression scale?** Determine raw counts vs. normalized/log values from source documentation. The numerical sample alone is not conclusive.
5. **For the TDA pilot:** choose ~5–10 well-represented target genes in K562, matched control cells, then inspect overlapping perturbations in RPE1. Avoid equating knockdown with complete gene knockout.

**Files to share for the next step:** `causalbench_repo_inventory.csv`, `causalbench_perturbation_cell_counts.csv`, and Step 4's column list. These permit an exact gene shortlist and preprocessing plan.